# 02 - Modelado

Entrenamos los modelos de regresión para predecir la variable `quality`. Usamos los conjuntos train/test generados en el notebook anterior.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
X_train = pd.read_csv('X_train.csv')
X_test  = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').squeeze()
y_test  = pd.read_csv('y_test.csv').squeeze()

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## Regresión Lineal

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
print('Entrenado.')

In [ ]:
print('--- Regresión Lineal ---')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lr)):.4f}')
print(f'MAE:  {mean_absolute_error(y_test, y_pred_lr):.4f}')
print(f'R²:   {r2_score(y_test, y_pred_lr):.4f}')

In [ ]:
# comparamos también con train para ver si hay sobreajuste
y_pred_lr_train = lr.predict(X_train)

print('Train vs Test:')
print(f'R² train: {r2_score(y_train, y_pred_lr_train):.4f}')
print(f'R² test:  {r2_score(y_test, y_pred_lr):.4f}')

## Modelos regularizados: Ridge y Lasso

En el EDA detectamos que algunas variables están correlacionadas entre sí, concretamente `free sulfur dioxide` y `total sulfur dioxide` tenían una correlación de ~0.67. Esto se conoce como multicolinealidad y es un problema para la regresión lineal: cuando dos variables aportan información muy parecida, el modelo tiene dificultades para asignar bien los coeficientes y puede dar valores inestables o inflados.

Para intentar mejorar esto probamos dos variantes de regresión que añaden una penalización a los coeficientes del modelo:

**Ridge** penaliza los coeficientes grandes sin llegar a eliminarlos. Hace que todos contribuyan pero de forma más moderada, lo que reduce la inestabilidad causada por la multicolinealidad. Esperamos que Ridge dé resultados similares o algo mejores que la regresión lineal simple, especialmente en términos de estabilidad.

**Lasso** aplica una penalización más agresiva que puede llevar algunos coeficientes exactamente a 0, eliminando directamente las variables menos útiles. Es interesante en nuestro caso porque si hay variables que apenas aportan a predecir la calidad, Lasso las descartará automáticamente. Esperamos que Lasso elimine alguna de las variables con baja correlación con `quality` que vimos en el EDA.

## Ridge

In [ ]:
alphas = [0.01, 0.1, 1, 10, 100]
ridge_cv = RidgeCV(alphas=alphas, cv=5)
ridge_cv.fit(X_train, y_train)

print(f'Mejor alpha: {ridge_cv.alpha_}')

In [ ]:
y_pred_ridge = ridge_cv.predict(X_test)

print('--- Ridge ---')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_ridge)):.4f}')
print(f'MAE:  {mean_absolute_error(y_test, y_pred_ridge):.4f}')
print(f'R²:   {r2_score(y_test, y_pred_ridge):.4f}')

## Lasso

In [ ]:
lasso_cv = LassoCV(alphas=alphas, cv=5, random_state=42, max_iter=10000)
lasso_cv.fit(X_train, y_train)

print(f'Mejor alpha: {lasso_cv.alpha_}')

In [ ]:
y_pred_lasso = lasso_cv.predict(X_test)

print('--- Lasso ---')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lasso)):.4f}')
print(f'MAE:  {mean_absolute_error(y_test, y_pred_lasso):.4f}')
print(f'R²:   {r2_score(y_test, y_pred_lasso):.4f}')

In [ ]:
# vemos que variables ha eliminado lasso (coeficiente = 0)
coefs_lasso = pd.Series(lasso_cv.coef_, index=X_train.columns)
print('Coeficientes Lasso:')
print(coefs_lasso.sort_values())

## Comparación de modelos

In [ ]:
resultados = pd.DataFrame({
    'Modelo': ['Regresión Lineal', 'Ridge', 'Lasso'],
    'RMSE': [
        np.sqrt(mean_squared_error(y_test, y_pred_lr)),
        np.sqrt(mean_squared_error(y_test, y_pred_ridge)),
        np.sqrt(mean_squared_error(y_test, y_pred_lasso))
    ],
    'MAE': [
        mean_absolute_error(y_test, y_pred_lr),
        mean_absolute_error(y_test, y_pred_ridge),
        mean_absolute_error(y_test, y_pred_lasso)
    ],
    'R²': [
        r2_score(y_test, y_pred_lr),
        r2_score(y_test, y_pred_ridge),
        r2_score(y_test, y_pred_lasso)
    ]
}).set_index('Modelo').round(4)

resultados

In [ ]:
# guardamos las predicciones para el notebook de evaluacion
pd.DataFrame({
    'y_test': y_test.values,
    'y_pred_lr': y_pred_lr,
    'y_pred_ridge': y_pred_ridge,
    'y_pred_lasso': y_pred_lasso
}).to_csv('predicciones.csv', index=False)

print('Predicciones guardadas en predicciones.csv')